**Reasoning**:
The first step is to import the necessary libraries (`requests`, `json`, `pandas`) and define the Jira API endpoint and parameters as per the instructions.



In [ ]:
import requests
import json
import pandas as pd

# 2. Define the Jira API endpoint URL
JIRA_API_URL = "https://issues.apache.org/jira/rest/api/2/search"

# 3. Create a dictionary of parameters for the API request
params = {
    "jql": "",  # Empty JQL to fetch all issues
    "maxResults": 500,
    "fields": "summary,description,status,created,updated,issuetype,priority,reporter,assignee,comment,labels"
}

print("Libraries imported, API endpoint and parameters defined.")

**Reasoning**:
Now that the API endpoint and parameters are defined, the next step is to send a GET request to the Jira API, check the response, parse the JSON, and store the extracted issues into a pandas DataFrame, as per the subtask instructions.



In [ ]:
import requests
import json
import pandas as pd
from datetime import datetime # Import datetime for date conversion

print("Sending GET request to Jira API...")
response = requests.get(JIRA_API_URL, params=params)

if response.status_code == 200:
    print("Request successful. Parsing JSON response...")
    data = response.json()
    issues = data.get('issues', [])

    # Prepare a list to hold processed issue data
    processed_issues = []
    for issue in issues:
        issue_fields = issue.get('fields', {})
        processed_issues.append({
            'id': issue.get('id'),
            'key': issue.get('key'),
            'summary': issue_fields.get('summary'),
            'description': issue_fields.get('description'),
            'status': (issue_fields.get('status') or {}).get('name'),
            'created': issue_fields.get('created'),
            'updated': issue_fields.get('updated'),
            'issuetype': (issue_fields.get('issuetype') or {}).get('name'),
            'priority': (issue_fields.get('priority') or {}).get('name'),
            'reporter': (issue_fields.get('reporter') or {}).get('displayName'),
            'assignee': (issue_fields.get('assignee') or {}).get('displayName'),
            'labels': issue_fields.get('labels'),
            'comments': [comment.get('body') for comment in issue_fields.get('comment', {}).get('comments', [])]
        })

    df_jira_issues = pd.DataFrame(processed_issues)

    # Convert 'created' and 'updated' columns to datetime objects
    df_jira_issues['created'] = pd.to_datetime(df_jira_issues['created'])
    df_jira_issues['updated'] = pd.to_datetime(df_jira_issues['updated'])

    print(f"Fetched {len(df_jira_issues)} issues.")
    print("Jira issues DataFrame created successfully.")
    print("First 5 rows of the DataFrame:")
    print(df_jira_issues.head())
else:
    print(f"Failed to fetch issues. Status code: {response.status_code}")
    print(f"Response: {response.text}")

**Reasoning**:
The next step is to preprocess the raw Jira issue data as per the instructions, which includes combining text fields, cleaning the text by removing HTML tags, special characters, stopwords, and then tokenizing it.



In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from bs4 import BeautifulSoup

# Download NLTK resources if not already downloaded
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
try:
    word_tokenize('test')
except LookupError:
    nltk.download('punkt')
try:
    # Explicitly download 'punkt_tab' as suggested by the error message
    nltk.data.find('tokenizers/punkt_tab/english.pickle')
except LookupError:
    nltk.download('punkt_tab')

# 2. Create a new column 'full_text' by combining 'summary' and 'description'
df_jira_issues['full_text'] = df_jira_issues['summary'].fillna('') + ' ' + \
                                df_jira_issues['description'].fillna('')

# 3. Define a function for text cleaning
def clean_text(text):
    if not isinstance(text, str):
        return []
    # a. Remove HTML tags
    soup = BeautifulSoup(text, 'html.parser')
    clean_text = soup.get_text(separator=' ')

    # b. Remove URLs and other non-alphanumeric characters
    clean_text = re.sub(r'http\S+|www\S+|https\S+', '', clean_text, flags=re.MULTILINE) # Remove URLs
    clean_text = re.sub(r'[^a-zA-Z\s]', '', clean_text) # Remove non-alphanumeric

    # c. Convert text to lowercase
    clean_text = clean_text.lower()

    # d. Tokenize the text
    tokens = word_tokenize(clean_text)

    # e. Remove stopwords
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word not in stop_words]

    return filtered_tokens

# 4. Apply this cleaning function to the 'full_text' column
df_jira_issues['processed_text'] = df_jira_issues['full_text'].apply(clean_text)

# 5. Display the head of the DataFrame with the new 'full_text' and 'processed_text' columns
print("Preprocessing complete. Displaying first 5 rows with new 'full_text' and 'processed_text' columns:")
print(df_jira_issues[['summary', 'description', 'full_text', 'processed_text']].head())

# Task
Analyze Jira issues by fetching up to 500 issues from `https://issues.apache.org/jira/rest/api/2/search`, preprocessing the text, generating TF-IDF vectors, clustering them into 5 groups using K-Means, extracting representative keywords for each cluster, and detecting potential duplicate issues using cosine similarity.

## Retrieve Jira Issues Script

### Subtask:
Generate a Python script to fetch a maximum of 500 Jira issues from the specified API endpoint (https://issues.apache.org/jira/rest/api/2/search) and store the raw data.


## Preprocess Jira Data Script

### Subtask:
Generate a Python script to preprocess the raw Jira issue data. This will include extracting relevant text fields (summary, description), cleaning the text (e.g., removing HTML tags, special characters, stopwords), and tokenization.


**Reasoning**:
The previous step successfully preprocessed the text data by cleaning and tokenizing it. The next logical step, as per the overall task, is to generate TF-IDF vectors from the `processed_text` column to prepare the data for clustering.



In [ ]:
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer

# Suppress the specific UserWarning regarding 'token_pattern'
warnings.filterwarnings("ignore", message="The parameter 'token_pattern' will not be used since 'tokenizer' is not None'", category=UserWarning)

# 6. Generate TF-IDF vectors from the 'processed_text' data
# Since 'processed_text' already contains lists of tokens, we use a custom tokenizer lambda x: x
# and set lowercase=False as the text is already lowercased.
tfidf_vectorizer = TfidfVectorizer(tokenizer=lambda x: x, preprocessor=lambda x: x, lowercase=False)
tfidf_matrix = tfidf_vectorizer.fit_transform(df_jira_issues['processed_text'])

print(f"TF-IDF matrix generated with shape: {tfidf_matrix.shape}")

# Optionally, reset warnings to default behavior after the operation if needed
# warnings.filterwarnings("default", category=UserWarning)

**Reasoning**:
The TF-IDF matrix has been successfully generated in the previous step. The next logical step, as per the overall task, is to cluster these vectors into 5 groups using K-Means.



In [ ]:
from sklearn.cluster import KMeans

# 7. Perform K-Means clustering
# Define the number of clusters (5 as per the task)
num_clusters = 5

# Initialize KMeans model. Using random_state for reproducibility.
# n_init='auto' is used to suppress a future warning about default value change.
kmeans_model = KMeans(n_clusters=num_clusters, init='k-means++', max_iter=300, random_state=42, n_init='auto')

# Fit the KMeans model to the TF-IDF matrix and get cluster assignments
df_jira_issues['cluster'] = kmeans_model.fit_predict(tfidf_matrix)

print(f"K-Means clustering completed with {num_clusters} clusters.")
print("Count of issues per cluster:")
print(df_jira_issues['cluster'].value_counts().sort_index())

print("First 5 rows of the DataFrame with cluster assignments:")
print(df_jira_issues[['summary', 'processed_text', 'cluster']].head())

**Reasoning**:
The K-Means clustering has been successfully completed, and issues are assigned to clusters. The next step, as per the overall task, is to extract representative keywords for each cluster using the TF-IDF vectorizer and the K-Means model's centroids.



In [ ]:
import numpy as np

# 8. Extract representative keywords for each cluster

# Get the feature names (words) from the TF-IDF vectorizer
feature_names = tfidf_vectorizer.get_feature_names_out()

# Get the centroids of the clusters
centroids = kmeans_model.cluster_centers_

# Number of top keywords to display for each cluster
num_top_keywords = 10

print(f"Top {num_top_keywords} keywords per cluster:")
for i in range(num_clusters):
    # Get the words with the highest TF-IDF scores for this cluster's centroid
    top_features_indices = centroids[i].argsort()[:-num_top_keywords - 1:-1]
    top_keywords = [feature_names[j] for j in top_features_indices]
    print(f"Cluster {i}: {', '.join(top_keywords)}")

**Reasoning**:
The previous step successfully extracted representative keywords for each cluster. The next logical step, as per the overall task, is to detect potential duplicate issues using cosine similarity. This involves calculating the cosine similarity matrix for the TF-IDF vectors and then identifying pairs of issues that have a similarity score above a certain threshold.



In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# 9. Detect potential duplicate issues using cosine similarity

# Calculate cosine similarity matrix
# Using the TF-IDF matrix generated earlier
cosine_sim_matrix = cosine_similarity(tfidf_matrix)

# Define a similarity threshold to identify potential duplicates
similarity_threshold = 0.8

print(f"Detecting potential duplicate issues with similarity > {similarity_threshold}:")

duplicate_pairs = []
# Iterate through the upper triangle of the similarity matrix to avoid duplicate pairs and self-comparison
for i in range(cosine_sim_matrix.shape[0]):
    for j in range(i + 1, cosine_sim_matrix.shape[1]):
        if cosine_sim_matrix[i, j] > similarity_threshold:
            issue1_key = df_jira_issues.loc[i, 'key']
            issue1_summary = df_jira_issues.loc[i, 'summary']
            issue2_key = df_jira_issues.loc[j, 'key']
            issue2_summary = df_jira_issues.loc[j, 'summary']
            duplicate_pairs.append({
                'issue1_key': issue1_key,
                'issue1_summary': issue1_summary,
                'issue2_key': issue2_key,
                'issue2_summary': issue2_summary,
                'similarity': cosine_sim_matrix[i, j]
            })

if duplicate_pairs:
    print(f"Found {len(duplicate_pairs)} potential duplicate pairs:")
    for pair in duplicate_pairs:
        print(f"- Issue '{pair['issue1_key']}' ('{pair['issue1_summary']}') and Issue '{pair['issue2_key']}' ('{pair['issue2_summary']}') with similarity: {pair['similarity']:.4f}")
else:
    print("No potential duplicate issues found above the specified similarity threshold.")

**Reasoning**:
The subtask requires generating a bar chart to visualize the distribution of issues across the 5 clusters. This involves importing `matplotlib.pyplot`, calculating cluster counts, and plotting the bar chart with appropriate labels and title.



## Phase 1 Summary: Jira Backlog Management Prototype

Phase 1 of the Jira backlog management prototype has been successfully completed, covering the following key stages:

1.  **Data Retrieval**: We successfully fetched a maximum of 500 Jira issues from the specified API endpoint (`https://issues.apache.org/jira/rest/api/2/search`). This data was loaded into a pandas DataFrame (`df_jira_issues`), providing a structured dataset for analysis.

2.  **Text Preprocessing**: The raw Jira issue data underwent a comprehensive preprocessing pipeline. This involved:
    *   Combining the 'summary' and 'description' fields into a new 'full_text' column.
    *   Cleaning the text by removing HTML tags, URLs, and other non-alphanumeric characters.
    *   Converting all text to lowercase.
    *   Tokenizing the cleaned text into individual words.
    *   Removing common English stopwords to focus on more significant terms.
    The processed text is stored in the 'processed_text' column, ready for numerical representation.

3.  **TF-IDF Vectorization**: Term Frequency-Inverse Document Frequency (TF-IDF) vectors were generated from the 'processed_text' column. This transformed the textual data into a numerical matrix (`tfidf_matrix`) of shape (500, 6842), where each row represents an issue and each column represents a unique term, with values indicating the importance of the term within that issue and across all issues.

4.  **Clustering with K-Means**: The TF-IDF vectors were clustered into 5 distinct groups using the K-Means algorithm. A new 'cluster' column was added to `df_jira_issues` to indicate each issue's cluster assignment. The distribution of issues across clusters was visualized using a bar chart, revealing the varying sizes and composition of the groups.

5.  **Keyword Extraction**: For each of the 5 clusters, the top 10 most representative keywords were extracted based on the cluster centroids in the TF-IDF space. These keywords provide a concise summary of the primary themes and topics discussed within each cluster, offering valuable insights into the nature of the issues.

6.  **Duplicate Detection**: Potential duplicate issues were identified by calculating the cosine similarity between the TF-IDF vectors of all issue pairs. A similarity threshold of 0.8 was applied, leading to the identification of 12 pairs of issues with high textual similarity, suggesting redundancy in the backlog.

### Preparation for Phase 2:

With Phase 1 complete, we have successfully established a foundation for intelligent Jira backlog management. The identified clusters, representative keywords, and potential duplicate issues provide actionable insights. Phase 2 will likely focus on leveraging these insights for practical applications, such as:

*   **Automated Issue Categorization**: Using the trained clustering model to automatically assign new incoming Jira issues to their appropriate clusters.
*   **Refinement of Duplicate Detection**: Further analysis and potential automation of the merging or linking of identified duplicate issues.
*   **Trend Analysis and Reporting**: Generating reports based on cluster distribution over time or specific keyword trends to inform project management and resource allocation.
*   **User Interface Integration**: Developing a user interface or integrating these functionalities into existing Jira workflows to provide real-time recommendations and insights to users.

## Final Task

### Subtask:
Summarize the completion of Phase 1, including the data retrieval, preprocessing, TF-IDF vectorization, clustering, keyword extraction, and duplicate detection, and prepare for Phase 2.


## Summary:

### Data Analysis Key Findings

*   **Jira Issue Retrieval**: Successfully fetched 500 Jira issues from the specified API endpoint, populating a pandas DataFrame named `df_jira_issues`. The data extraction process involved handling potential `None` values for nested fields to prevent errors.
*   **Text Preprocessing**: Combined 'summary' and 'description' fields, cleaned the text by removing HTML tags, URLs, and special characters, lowercased, and tokenized the text. A `LookupError` for the NLTK `punkt_tab` resource was resolved by explicitly downloading it.
*   **TF-IDF Vectorization**: Generated a TF-IDF matrix with a shape of (500, 6842) from the processed text, representing the numerical features of each issue. A `UserWarning` during vectorization was successfully suppressed.
*   **K-Means Clustering**: Grouped the 500 issues into 5 clusters. The distribution showed varying sizes, with Cluster 3 containing the most issues (286) and Cluster 0 containing the fewest (42).
*   **Keyword Extraction**: Identified the top 10 representative keywords for each cluster, providing insights into their primary themes. For instance, Cluster 0 was characterized by keywords like 'support', 'auth', and 'apis', while Cluster 1 featured 'cve', 'netty', and 'jetty'.
*   **Duplicate Detection**: Identified 12 potential duplicate issue pairs by calculating cosine similarity between issue TF-IDF vectors and applying a threshold of 0.8. An example includes issues 'ZOOKEEPER-4948' and 'ZOOKEEPER-4947', which showed a perfect similarity of 1.0000.

### Insights or Next Steps

*   The identified clusters and their representative keywords can be used to categorize new Jira issues automatically, improving triage and routing efficiency.
*   The detected duplicate issues warrant further investigation to merge redundant entries, thereby streamlining issue tracking and reducing developer effort.


# Task
Generate a bar chart to visualize the distribution of issues across the 5 clusters. The x-axis will represent the cluster IDs, and the y-axis will represent the number of issues in each cluster. The plot should include appropriate labels and a title. Conclude Phase 1 of the Jira backlog management prototype by summarizing the clustering and duplicate detection results and preparing for Phase 2.

## Generate Bar Chart

### Subtask:
Create a bar chart to visualize the distribution of issues across the 5 clusters. The x-axis will represent the cluster IDs, and the y-axis will represent the number of issues in each cluster. The plot should include appropriate labels and a title.


## Final Task

### Subtask:
Conclude Phase 1 of the Jira backlog management prototype by summarizing the clustering and duplicate detection results and preparing for Phase 2.


## Summary:

### Q&A
1.  **How was the distribution of issues across the 5 clusters visualized?**
    A bar chart was generated, displaying 'Cluster ID' on the x-axis and 'Number of Issues' on the y-axis, with the title 'Distribution of Issues Across Clusters'.
2.  **What were the key results of the clustering and duplicate detection in Phase 1?**
    Phase 1 successfully clustered 500 Jira issues into 5 distinct groups using K-Means, extracted the top 10 keywords for each cluster, and identified 12 pairs of potential duplicate issues based on a cosine similarity threshold of 0.8.

### Data Analysis Key Findings
*   **Data Acquisition and Preprocessing**: A total of 500 Jira issues were retrieved, preprocessed (cleaned, tokenized, stop words removed), and transformed into a numerical TF-IDF matrix of shape (500, 6842).
*   **Clustering Results**: The issues were successfully clustered into 5 distinct groups using K-Means, with a bar chart visualizing the distribution of issues across these clusters.
*   **Keyword Extraction**: For each of the 5 clusters, the top 10 most representative keywords were extracted, offering insights into the themes within each cluster.
*   **Duplicate Detection**: Utilizing a cosine similarity threshold of 0.8 on TF-IDF vectors, 12 pairs of issues were identified as potential duplicates, indicating redundancy in the backlog.

### Insights or Next Steps
*   Leverage the trained clustering model to automatically categorize new incoming Jira issues, improving backlog organization.
*   Further refine the duplicate detection process, potentially automating the merging or linking of identified duplicate issues for enhanced efficiency.


**Reasoning**:
The subtask is to create a dummy 'release_notes.txt' file with placeholder content. This involves defining the content as a string and then writing it to a file, followed by a confirmation message.



In [ ]:
release_notes_content = """
Release Notes - Version 3.8.1
Release Date: May 1, 2026
*   Bug Fix: Addressed critical security vulnerability (CVE-2026-XXXX).
*   Improvement: Enhanced performance of client connection handling.
*   Feature: Introduced new metrics for Prometheus integration.

Release Notes - Version 3.8.0
Release Date: April 15, 2026
*   Major Feature: Implemented SASL authentication for client connections.
*   Bug Fix: Resolved an issue with session expiration during network partitions.
*   Improvement: Updated dependency `netty-tcnative` to 2.0.60.Final.

Release Notes - Version 3.7.1
Release Date: March 10, 2026
*   Bug Fix: Fixed an NPE in `ConnectionMetricsTest`.
*   Improvement: Optimized leader election process for faster failover.
*   Feature: Added support for auto-reloading client key/trust stores.

Release Notes - Version 3.7.0
Release Date: February 20, 2026
*   Feature: Introduced Admin Server APIs for enhanced management.
*   Improvement: Refactored logging to use SLF4J and Logback.
*   Bug Fix: Corrected issue with data directory size reporting.

Release Notes - Version 3.6.3
Release Date: January 5, 2026
*   Security Fix: Addressed minor vulnerabilities in third-party libraries.
*   Improvement: Better handling of dynamic configuration file updates.
*   Feature: Expanded unit test coverage for consistency semantics.
"""

with open('release_notes.txt', 'w') as f:
    f.write(release_notes_content)

print("Created 'release_notes.txt' with dummy release notes content.")

**Reasoning**:
The subtask requires creating a dummy 'project_context.txt' file with placeholder content describing the project's goals, descriptions, process, and workflow. This involves defining the content as a string and then writing it to a file, followed by a confirmation message.



In [ ]:
project_context_content = """
Project Context: Jira Backlog Management Prototype

Goals:
*   Improve efficiency in Jira backlog management.
*   Automate identification of duplicate issues.
*   Facilitate intelligent clustering and categorization of issues.
*   Provide actionable insights for project managers and development teams.

Description:
This project aims to develop a prototype system for advanced Jira backlog management using natural language processing (NLP) and machine learning techniques. It involves fetching Jira issues, preprocessing their textual content, and applying clustering algorithms to group similar issues. Additionally, it identifies potential duplicate issues to streamline the backlog and reduce redundancy. The system also extracts key themes from clusters to offer high-level insights.

Process:
1.  **Data Retrieval**: Fetch up to 500 Jira issues via the Jira REST API.
2.  **Text Preprocessing**: Clean and tokenize issue summaries and descriptions.
3.  **Feature Engineering**: Convert text into numerical TF-IDF vectors.
4.  **Clustering**: Apply K-Means to group issues into 5 clusters.
5.  **Keyword Extraction**: Identify representative keywords for each cluster.
6.  **Duplicate Detection**: Calculate cosine similarity to find highly similar issues.
7.  **Analysis & Reporting**: Summarize findings and visualize distributions.

Workflow:
*   **Initial Setup**: Define API credentials and endpoint.
*   **Execution**: Run the Python script to perform data retrieval and analysis.
*   **Review Results**: Examine cluster assignments, keywords, and duplicate pairs.
*   **Actionable Insights**: Utilize the generated insights to refine Jira backlog, assign issues, or merge duplicates.
*   **Iteration**: Continuously refine models and parameters based on feedback and new data.
"""

with open('project_context.txt', 'w') as f:
    f.write(project_context_content)

print("Created 'project_context.txt' with dummy project context content.")

# Task
Create a dummy `release_notes.txt` file with placeholder content relevant to past 4-6 releases for use as a knowledge base.

## Create Dummy Release Notes RAG

### Subtask:
Create a dummy 'release_notes.txt' file with placeholder content relevant to past 4-6 releases.


## Create Dummy Project Context RAG

### Subtask:
Create a dummy 'project_context.txt' file with placeholder content describing the project's goals, descriptions, process, and workflow.


**Reasoning**:
The subtask requires creating a dummy '2026_theme.txt' file with placeholder content outlining the current year's increment purpose and themes for FY26Q1-Q4. This involves defining the content as a string and then writing it to a file, followed by a confirmation message.



In [ ]:
theme_2026_content = """
2026 Themes and Increment Purpose

Overall Increment Purpose:
To enhance the stability, scalability, and security of our core systems while introducing innovative features that improve user experience and operational efficiency. Focus will be on reducing technical debt, optimizing performance, and preparing for future growth.

FY26Q1 Themes (January - March 2026):
*   **Theme 1: Core System Hardening**: Focus on identifying and patching critical vulnerabilities, improving error handling, and implementing robust logging and monitoring solutions.
*   **Theme 2: Performance Optimization**: Initiatives to reduce latency, optimize database queries, and improve overall system responsiveness for high-traffic operations.
*   **Theme 3: User Authentication Revamp**: Upgrade authentication mechanisms to support modern standards and improve security posture.

FY26Q2 Themes (April - June 2026):
*   **Theme 1: Scalability Enhancements**: Develop and deploy solutions for dynamic resource allocation, load balancing, and horizontal scaling to support increased user loads.
*   **Theme 2: Data Integrity & Governance**: Implement stricter data validation rules, introduce data lineage tracking, and improve data backup and recovery processes.
*   **Theme 3: API Modernization**: Refactor and standardize existing APIs, and introduce new GraphQL endpoints for more flexible data access.

FY26Q3 Themes (July - September 2026):
*   **Theme 1: Advanced Analytics Integration**: Integrate with new analytics platforms to provide deeper insights into user behavior and system performance.
*   **Theme 2: AI/ML Feature Development**: Pilot and develop initial AI/ML-driven features, such as intelligent recommendations or automated anomaly detection.
*   **Theme 3: Developer Experience Improvement**: Streamline CI/CD pipelines, improve documentation, and develop internal tools to boost developer productivity.

FY26Q4 Themes (October - December 2026):
*   **Theme 1: Cloud Migration Readiness**: Assess and prepare core applications for potential migration to a cloud-native architecture, focusing on containerization and microservices.
*   **Theme 2: Compliance & Regulatory Adherence**: Ensure all systems and processes meet evolving industry compliance standards and regulatory requirements.
*   **Theme 3: Strategic Partnership Integrations**: Build and refine integrations with key third-party partners to expand ecosystem capabilities and value proposition.
"""

with open('2026_theme.txt', 'w') as f:
    f.write(theme_2026_content)

print("Created '2026_theme.txt' with dummy 2026 themes content.")

## Create Dummy 2026 Theme RAG

### Subtask:
Create a dummy '2026_theme.txt' file with placeholder content outlining the current year's increment purpose and themes for FY26Q1-Q4. This will serve as a knowledge base for the LLM.


**Reasoning**:
The subtask requires generating descriptive names for each cluster using an LLM, leveraging the RAG context files and previously extracted cluster keywords. This step will import necessary libraries, initialize the LLM, load context files, construct a prompt template, and then iterate through each cluster to invoke the LLM for name generation.



In [ ]:
get_ipython().system('pip install langchain_google_genai')
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
import json
import re
import time


# Load API key from Colab Secrets
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')

# 1. Initialize the ChatGoogleGenAI model
llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0.3, api_key=api_key)


# 2. Load the content of the RAG context files
with open('release_notes.txt', 'r') as f:
    release_notes_context = f.read()

with open('project_context.txt', 'r') as f:
    project_context = f.read()

with open('2026_theme.txt', 'r') as f:
    theme_2026_context = f.read()

# 3. Construct a prompt template for enhanced cluster details
cluster_enhancement_prompt_template = PromptTemplate(
    input_variables=["release_notes_context", "project_context", "theme_2026_context", "cluster_keywords", "cluster_id"],
    template=(
        "You are an expert in Jira issue analysis and project management. Your task is to provide a comprehensive analysis for a Jira issue cluster, including a descriptive name, its alignment with 2026 themes, a detailed description, and actionable Project Manager recommendations.\n\n"
        "Here is some relevant project context (use for overall understanding):\n{project_context}\n\n"
        "Here are past release notes and features (use for historical context and identifying trends):\n{release_notes_context}\n\n"
        "Here are the 2026 Themes (prioritize for theme alignment and forward-looking recommendations):\n{theme_2026_context}\n\n"
        "The following are the top keywords for Cluster {cluster_id}:\n{cluster_keywords}\n\n"
        "Based on this information, provide the following in JSON format:\n"
        "{{~\n"
        "  \"name\": \"[A concise and descriptive name for Cluster {cluster_id} (5-8 words)]\",\\n"
        "  \"theme_alignment\": \"[How does this cluster align with or impact the 2026 themes? (2-3 sentences)]\",\\n"
        "  \"description\": \"[A detailed description of the cluster's core issues and themes based on keywords and context. (3-5 sentences)]\",\\n"
        "  \"pm_recommendations\": \"[Actionable recommendations for a Project Manager to address this cluster, considering project context, release notes, and 2026 themes. (3-4 sentences)]\"\\n"
        "}}\n"
    )
)

# Store generated cluster details
enhanced_cluster_details = {}

print("Generating enhanced details for each cluster...")

# 4. Iterate through each cluster and generate details
for i in range(num_clusters):
    # Get the keywords for the current cluster
    top_features_indices = centroids[i].argsort()[:-num_top_keywords - 1:-1]
    current_cluster_keywords = [feature_names[j] for j in top_features_indices]
    cluster_keywords_str = ", ".join(current_cluster_keywords)

    # Format the prompt
    formatted_prompt = cluster_enhancement_prompt_template.format(
        release_notes_context=release_notes_context,
        project_context=project_context,
        theme_2026_context=theme_2026_context,
        cluster_keywords=cluster_keywords_str,
        cluster_id=i
    )

    # Invoke the LLM
    response = llm.invoke(formatted_prompt)
    try:
        # Extract JSON from markdown code block if present using regex
        content = response.content.strip()
        match = re.search(r'```json\s*(.*?)\s*```', content, re.DOTALL)
        if match:
            json_str = match.group(1).strip()
        else:
            # Fallback if no markdown block, assume content is pure JSON
            json_str = content

        cluster_output = json.loads(json_str)
        enhanced_cluster_details[i] = cluster_output
        print(f"Cluster {i} Name: {cluster_output['name']}")
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON for Cluster {i}: {e} -- Content: {response.content}")
        enhanced_cluster_details[i] = {'name': 'Error', 'theme_alignment': 'Error', 'description': 'Error', 'pm_recommendations': 'Error'}

    # Add a delay to avoid hitting API rate limits
    time.sleep(60) # Wait for 60 seconds before the next request

print("\nGenerated enhanced cluster details:")
for cluster_id, details in enhanced_cluster_details.items():
    print(f"Cluster {cluster_id}: {details['name']}")
    print(f"  2026 Theme Alignment: {details['theme_alignment']}")
    print(f"  Description: {details['description']}")
    print(f"  PM Recommendations: {details['pm_recommendations']}\n")

# Update cluster_names for backward compatibility with existing summary if not immediately replaced
cluster_names = {cid: details['name'] for cid, details in enhanced_cluster_details.items()}

## Enhance Cluster Names with LLM

### Subtask:
Utilize an LLM, along with the created RAG contexts and previously extracted cluster keywords, to generate more descriptive and contextually relevant names for each of the 5 Jira issue clusters.


**Reasoning**:
The previous code failed because the `langchain_google_genai` module was not found. This indicates that the package needs to be installed. I will add a `!pip install` command to install it before re-running the code.



### LLM-Generated Enhanced Cluster Names:

*   **Cluster 0:** Admin Server APIs, Authentication, and Prometheus Metrics
*   **Cluster 1:** Security Vulnerability Fixes and Dependency Upgrades
*   **Cluster 2:** ZooKeeper Client Connectivity, Sessions, and Security
*   **Cluster 3:** ZooKeeper Client/Server Errors and Stability Issues
*   **Cluster 4:** Core System Dependency and Logging Updates

This completes the LLM-enhanced task of generating more descriptive cluster names. Please let me know if you would like to proceed with any further analysis or tasks for Phase 2!

## Enhance Duplicate Insights with LLM

### Subtask:
Leverage the LLM to generate more detailed insights for each identified duplicate issue pair, explaining the potential reasons for duplication and suggesting actions.

In [ ]:
get_ipython().system('pip install langchain_google_genai') # Ensure installation
from datetime import datetime, timedelta, timezone
import json
import re
import time
import pandas as pd
from langchain_core.prompts import PromptTemplate
from google.colab import userdata # Needed for api_key
from langchain_google_genai import ChatGoogleGenerativeAI # Needed for llm
import os # Import os for file existence checks

print("Generating enhanced insights for duplicate issue pairs...")

enhanced_duplicate_insights = []

# Ensure API key is loaded from Colab Secrets
api_key = userdata.get('GEMINI_API_KEY')

# Define content for context files
release_notes_content = """
Release Notes - Version 3.8.1
Release Date: May 1, 2026
*   Bug Fix: Addressed critical security vulnerability (CVE-2026-XXXX).
*   Improvement: Enhanced performance of client connection handling.
*   Feature: Introduced new metrics for Prometheus integration.

Release Notes - Version 3.8.0
Release Date: April 15, 2026
*   Major Feature: Implemented SASL authentication for client connections.
*   Bug Fix: Resolved an issue with session expiration during network partitions.
*   Improvement: Updated dependency `netty-tcnative` to 2.0.60.Final.

Release Notes - Version 3.7.1
Release Date: March 10, 2026
*   Bug Fix: Fixed an NPE in `ConnectionMetricsTest`.
*   Improvement: Optimized leader election process for faster failover.
*   Feature: Added support for auto-reloading client key/trust stores.

Release Notes - Version 3.7.0
Release Date: February 20, 2026
*   Feature: Introduced Admin Server APIs for enhanced management.
*   Improvement: Refactored logging to use SLF4J and Logback.
*   Bug Fix: Corrected issue with data directory size reporting.

Release Notes - Version 3.6.3
Release Date: January 5, 2026
*   Security Fix: Addressed minor vulnerabilities in third-party libraries.
*   Improvement: Better handling of dynamic configuration file updates.
*   Feature: Expanded unit test coverage for consistency semantics.
"""

project_context_content = """
Project Context: Jira Backlog Management Prototype

Goals:
*   Improve efficiency in Jira backlog management.
*   Automate identification of duplicate issues.
*   Facilitate intelligent clustering and categorization of issues.
*   Provide actionable insights for project managers and development teams.

Description:
This project aims to develop a prototype system for advanced Jira backlog management using natural language processing (NLP) and machine learning techniques. It involves fetching Jira issues, preprocessing their textual content, and applying clustering algorithms to group similar issues. Additionally, it identifies potential duplicate issues to streamline the backlog and reduce redundancy. The system also extracts key themes from clusters to offer high-level insights.

Process:
1.  **Data Retrieval**: Fetch up to 500 Jira issues via the Jira REST API.
2.  **Text Preprocessing**: Clean and tokenize issue summaries and descriptions.
3.  **Feature Engineering**: Convert text into numerical TF-IDF vectors.
4.  **Clustering**: Apply K-Means to group issues into 5 clusters.
5.  **Keyword Extraction**: Identify representative keywords for each cluster.
6.  **Duplicate Detection**: Calculate cosine similarity to find highly similar issues.
7.  **Analysis & Reporting**: Summarize findings and visualize distributions.

Workflow:
*   **Initial Setup**: Define API credentials and endpoint.
*   **Execution**: Run the Python script to perform data retrieval and analysis.
*   **Review Results**: Examine cluster assignments, keywords, and duplicate pairs.
*   **Actionable Insights**: Utilize the generated insights to refine Jira backlog, assign issues, or merge duplicates.
*   **Iteration**: Continuously refine models and parameters based on feedback and new data.
"""

theme_2026_content = """
2026 Themes and Increment Purpose

Overall Increment Purpose:
To enhance the stability, scalability, and security of our core systems while introducing innovative features that improve user experience and operational efficiency. Focus will be on reducing technical debt, optimizing performance, and preparing for future growth.

FY26Q1 Themes (January - March 2026):
*   **Theme 1: Core System Hardening**: Focus on identifying and patching critical vulnerabilities, improving error handling, and implementing robust logging and monitoring solutions.
*   **Theme 2: Performance Optimization**: Initiatives to reduce latency, optimize database queries, and improve overall system responsiveness for high-traffic operations.
*   **Theme 3: User Authentication Revamp**: Upgrade authentication mechanisms to support modern standards and improve security posture.

FY26Q2 Themes (April - June 2026):
*   **Theme 1: Scalability Enhancements**: Develop and deploy solutions for dynamic resource allocation, load balancing, and horizontal scaling to support increased user loads.
*   **Theme 2: Data Integrity & Governance**: Implement stricter data validation rules, introduce data lineage tracking, and improve data backup and recovery processes.
*   **Theme 3: API Modernization**: Refactor and standardize existing APIs, and introduce new GraphQL endpoints for more flexible data access.

FY26Q3 Themes (July - September 2026):
*   **Theme 1: Advanced Analytics Integration**: Integrate with new analytics platforms to provide deeper insights into user behavior and system performance.
*   **Theme 2: AI/ML Feature Development**: Pilot and develop initial AI/ML-driven features, suchs as intelligent recommendations or automated anomaly detection.
*   **Theme 3: Developer Experience Improvement**: Streamline CI/CD pipelines, improve documentation, and develop internal tools to boost developer productivity.

FY26Q4 Themes (October - December 2026):
*   **Theme 1: Cloud Migration Readiness**: Assess and prepare core applications for potential migration to a cloud-native architecture, focusing on containerization and microservices.
*   **Theme 2: Compliance & Regulatory Adherence**: Ensure all systems and processes meet evolving industry compliance standards and regulatory requirements.
*   **Theme 3: Strategic Partnership Integrations**: Build and refine integrations with key third-party partners to expand ecosystem capabilities and value proposition.
"""

# Create context files (overwriting if they exist to ensure content is correct)
with open('release_notes.txt', 'w') as f:
    f.write(release_notes_content)

with open('project_context.txt', 'w') as f:
    f.write(project_context_content)

with open('2026_theme.txt', 'w') as f:
    f.write(theme_2026_content)

# Load the content of the RAG context files into variables
release_notes_context = release_notes_content
project_context = project_context_content
theme_2026_context = theme_2026_content

# Initialize the ChatGoogleGenAI model
llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0.3, api_key=api_key)

# Function to categorize issue age
def categorize_age(updated_date, current_date):
    if pd.isnull(updated_date):
        return "Unknown"
    age_days = (current_date - updated_date).days
    if age_days < 30:
        return "Active (< 30 days)"
    elif 30 <= age_days < 60:
        return "Recent (30-60 days)"
    elif 60 <= age_days < 180:
        return "Aging (> 60 days)"
    elif 180 <= age_days < 365:
        return "Stale (> 180 days)"
    else:
        return "Very Stale (> 365 days)"

current_date = pd.Timestamp.now(tz=timezone.utc)

duplicate_insight_prompt_template = PromptTemplate(
    input_variables=["release_notes_context", "project_context", "theme_2026_context",
                     "issue1_key", "issue1_summary", "issue1_description", "issue1_age_category",
                     "issue2_key", "issue2_summary", "issue2_description", "issue2_age_category", "similarity"],
    template=(
        "You are an expert in Jira issue analysis. Based on the provided context, analyze the following two Jira issues and identify if they are duplicates. Provide detailed reasoning and actionable recommendations.\n\n"
        "Project Context (use for overall understanding):\n{project_context}\n\n"
        "Release Notes Context (PRIORITIZE for historical fixes and features):\n{release_notes_context}\n\n"
        "2026 Themes Context (PRIORITIZE for strategic alignment and future impact):\n{theme_2026_context}\n\n"
        "Issue 1 Key: {issue1_key}\n"
        "Issue 1 Summary: {issue1_summary}\n"
        "Issue 1 Description: {issue1_description}\n"
        "Issue 1 Age Category: {issue1_age_category}\n\n"
        "Issue 2 Key: {issue2_key}\n"
        "Issue 2 Summary: {issue2_summary}\n"
        "Issue 2 Description: {issue2_description}\n"
        "Issue 2 Age Category: {issue2_age_category}\n\n"
        "Similarity Score: {similarity:.4f}\n\n"
        "Based on the above, provide your analysis in JSON format with two keys: `llm_reasoning` and `llm_recommendations`.\n"
        "{{~\n"
        "  \"llm_reasoning\": \"[Your detailed reasoning on why these might be duplicates, considering all contexts including age categories.]\",\\n"
        "  \"llm_recommendations\": \"[Actionable recommendations for a Project Manager to resolve this duplicate pair, prioritizing insights from release notes and 2026 themes.]\"\\n"
        "}}\n"
    )
)

# Ensure df_jira_issues and duplicate_pairs are defined for a meaningful run
if 'df_jira_issues' not in locals() and 'df_jira_issues' not in globals():
    print("Warning: df_jira_issues is not defined. Please run previous cells to fetch and preprocess Jira issues.")
    df_jira_issues = pd.DataFrame(columns=['key', 'summary', 'description', 'updated'])

if 'duplicate_pairs' not in locals() and 'duplicate_pairs' not in globals():
    print("Warning: duplicate_pairs is not defined. Please run previous cells to identify duplicate issues.")
    duplicate_pairs = []

for i, pair in enumerate(duplicate_pairs):
    issue1_key = pair['issue1_key']
    issue1_summary = pair['issue1_summary']
    issue2_key = pair['issue2_key']
    issue2_summary = pair['issue2_summary']
    similarity = pair['similarity']

    # Fetch full description and updated date
    # df_jira_issues needs to be defined from previous cells for this to work correctly.
    issue1_data = df_jira_issues[df_jira_issues['key'] == issue1_key].iloc[0]
    issue1_description = issue1_data['description'] if issue1_data['description'] is not None else "No description provided."
    issue1_updated_date = issue1_data['updated']
    issue1_age_category = categorize_age(issue1_updated_date, current_date)

    issue2_data = df_jira_issues[df_jira_issues['key'] == issue2_key].iloc[0]
    issue2_description = issue2_data['description'] if issue2_data['description'] is not None else "No description provided."
    issue2_updated_date = issue2_data['updated']
    issue2_age_category = categorize_age(issue2_updated_date, current_date)

    formatted_insight_prompt = duplicate_insight_prompt_template.format(
        release_notes_context=release_notes_context,
        project_context=project_context,
        theme_2026_context=theme_2026_context,
        issue1_key=issue1_key,
        issue1_summary=issue1_summary,
        issue1_description=issue1_description,
        issue1_age_category=issue1_age_category,
        issue2_key=issue2_key,
        issue2_summary=issue2_summary,
        issue2_description=issue2_description,
        issue2_age_category=issue2_age_category,
        similarity=similarity
    )

    insight_response = llm.invoke(formatted_insight_prompt)
    try:
        content = insight_response.content.strip()
        match = re.search(r'```json\s*(.*?)\s*```', content, re.DOTALL)
        if match:
            json_str = match.group(1).strip()
        else:
            json_str = content

        insight_output = json.loads(json_str)
        enhanced_duplicate_insights.append({
            'issue1_key': issue1_key,
            'issue1_age_category': issue1_age_category,
            'issue2_key': issue2_key,
            'issue2_age_category': issue2_age_category,
            'similarity': similarity,
            'llm_reasoning': insight_output.get('llm_reasoning', 'No reasoning provided. líquidas.'),
            'llm_recommendations': insight_output.get('llm_recommendations', 'No recommendations provided.')
        })
        print(f"Pair {i+1}: {issue1_key} & {issue2_key} - Insight generated.")
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON for pair {issue1_key} & {issue2_key}: {e} -- Content: {insight_response.content}")
        enhanced_duplicate_insights.append({
            'issue1_key': issue1_key,
            'issue1_age_category': issue1_age_category,
            'issue2_key': issue2_key,
            'issue2_age_category': issue2_age_category,
            'similarity': similarity,
            'llm_reasoning': 'Error processing LLM response.',
            'llm_recommendations': 'Error processing LLM response.'
        })

    time.sleep(60)

print("\nEnhanced Duplicate Insights:")
for insight in enhanced_duplicate_insights:
    print(f"- Issues {insight['issue1_key']} ({insight['issue1_age_category']}) and {insight['issue2_key']} ({insight['issue2_age_category']}) (Similarity: {insight['similarity']:.4f}):\n  Reasoning: {insight['llm_reasoning']}\n  Recommendations: {insight['llm_recommendations']}\n")

In [ ]:
import os # Import os for file operations (reading context files)
from langchain_google_genai import ChatGoogleGenerativeAI # Import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from google.colab import userdata # Import userdata for API key

print("Generating Executive Summary...")

# Ensure API key is loaded from Colab Secrets
api_key = userdata.get('GEMINI_API_KEY')

# Define content for context files
release_notes_content = """
Release Notes - Version 3.8.1
Release Date: May 1, 2026
*   Bug Fix: Addressed critical security vulnerability (CVE-2026-XXXX).
*   Improvement: Enhanced performance of client connection handling.
*   Feature: Introduced new metrics for Prometheus integration.

Release Notes - Version 3.8.0
Release Date: April 15, 2026
*   Major Feature: Implemented SASL authentication for client connections.
*   Bug Fix: Resolved an issue with session expiration during network partitions.
*   Improvement: Updated dependency `netty-tcnative` to 2.0.60.Final.

Release Notes - Version 3.7.1
Release Date: March 10, 2026
*   Bug Fix: Fixed an NPE in `ConnectionMetricsTest`.
*   Improvement: Optimized leader election process for faster failover.
*   Feature: Added support for auto-reloading client key/trust stores.

Release Notes - Version 3.7.0
Release Date: February 20, 2026
*   Feature: Introduced Admin Server APIs for enhanced management.
*   Improvement: Refactored logging to use SLF4J and Logback.
*   Bug Fix: Corrected issue with data directory size reporting.

Release Notes - Version 3.6.3
Release Date: January 5, 2026
*   Security Fix: Addressed minor vulnerabilities in third-party libraries.
*   Improvement: Better handling of dynamic configuration file updates.
*   Feature: Expanded unit test coverage for consistency semantics.
"""

project_context_content = """
Project Context: Jira Backlog Management Prototype

Goals:
*   Improve efficiency in Jira backlog management.
*   Automate identification of duplicate issues.
*   Facilitate intelligent clustering and categorization of issues.
*   Provide actionable insights for project managers and development teams.

Description:
This project aims to develop a prototype system for advanced Jira backlog management using natural language processing (NLP) and machine learning techniques. It involves fetching Jira issues, preprocessing their textual content, and applying clustering algorithms to group similar issues. Additionally, it identifies potential duplicate issues to streamline the backlog and reduce redundancy. The system also extracts key themes from clusters to offer high-level insights.

Process:
1.  **Data Retrieval**: Fetch up to 500 Jira issues via the Jira REST API.
2.  **Text Preprocessing**: Clean and tokenize issue summaries and descriptions.
3.  **Feature Engineering**: Convert text into numerical TF-IDF vectors.
4.  **Clustering**: Apply K-Means to group issues into 5 clusters.
5.  **Keyword Extraction**: Identify representative keywords for each cluster.
6.  **Duplicate Detection**: Calculate cosine similarity to find highly similar issues.
7.  **Analysis & Reporting**: Summarize findings and visualize distributions.

Workflow:
*   **Initial Setup**: Define API credentials and endpoint.
*   **Execution**: Run the Python script to perform data retrieval and analysis.
*   **Review Results**: Examine cluster assignments, keywords, and duplicate pairs.
*   **Actionable Insights**: Utilize the generated insights to refine Jira backlog, assign issues, or merge duplicates.
*   **Iteration**: Continuously refine models and parameters based on feedback and new data.
"""

theme_2026_content = """
2026 Themes and Increment Purpose

Overall Increment Purpose:
To enhance the stability, scalability, and security of our core systems while introducing innovative features that improve user experience and operational efficiency. Focus will be on reducing technical debt, optimizing performance, and preparing for future growth.

FY26Q1 Themes (January - March 2026):
*   **Theme 1: Core System Hardening**: Focus on identifying and patching critical vulnerabilities, improving error handling, and implementing robust logging and monitoring solutions.
*   **Theme 2: Performance Optimization**: Initiatives to reduce latency, optimize database queries, and improve overall system responsiveness for high-traffic operations.
*   **Theme 3: User Authentication Revamp**: Upgrade authentication mechanisms to support modern standards and improve security posture.

FY26Q2 Themes (April - June 2026):
*   **Theme 1: Scalability Enhancements**: Develop and deploy solutions for dynamic resource allocation, load balancing, and horizontal scaling to support increased user loads.
*   **Theme 2: Data Integrity & Governance**: Implement stricter data validation rules, introduce data lineage tracking, and improve data backup and recovery processes.
*   **Theme 3: API Modernization**: Refactor and standardize existing APIs, and introduce new GraphQL endpoints for more flexible data access.

FY26Q3 Themes (July - September 2026):
*   **Theme 1: Advanced Analytics Integration**: Integrate with new analytics platforms to provide deeper insights into user behavior and system performance.
*   **Theme 2: AI/ML Feature Development**: Pilot and develop initial AI/ML-driven features, such as intelligent recommendations or automated anomaly detection.
*   **Theme 3: Developer Experience Improvement**: Streamline CI/CD pipelines, improve documentation, and develop internal tools to boost developer productivity.

FY26Q4 Themes (October - December 2026):
*   **Theme 1: Cloud Migration Readiness**: Assess and prepare core applications for potential migration to a cloud-native architecture, focusing on containerization and microservices.
*   **Theme 2: Compliance & Regulatory Adherence**: Ensure all systems and processes meet evolving industry compliance standards and regulatory requirements.
*   **Theme 3: Strategic Partnership Integrations**: Build and refine integrations with key third-party partners to expand ecosystem capabilities and value proposition.
"""

# Create context files (overwriting if they exist to ensure content is correct)
with open('release_notes.txt', 'w') as f:
    f.write(release_notes_content)

with open('project_context.txt', 'w') as f:
    f.write(project_context_content)

with open('2026_theme.txt', 'w') as f:
    f.write(theme_2026_content)

# Load the content of the RAG context files into variables
release_notes_context = release_notes_content
project_context = project_context_content
theme_2026_context = theme_2026_content

# Initialize the ChatGoogleGenAI model
llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0.3, api_key=api_key)

executive_summary_prompt_template = PromptTemplate(
    input_variables=["project_context", "theme_2026_context", "enhanced_cluster_details", "enhanced_duplicate_insights"],
    template=(
        "You are an expert in Jira backlog analysis and strategic reporting. Generate a concise executive summary (3-5 paragraphs) of the Jira issue analysis, focusing on key findings from cluster analysis and duplicate issue detection. Emphasize the alignment with and impact on the 2026 themes. Provide actionable, high-level recommendations for project management.\n\n"
        "## Project Context:\n{project_context}\n\n"
        "## 2026 Themes (PRIORITIZE for strategic alignment and impact):\n{theme_2026_context}\n\n"
        "## Cluster Analysis Summary:\n"
        "Based on the clustering of Jira issues, here are the enhanced details for each cluster:\n"
        "{enhanced_cluster_details}\n\n"
        "## Duplicate Issue Analysis Summary:\n"
        "Here are the insights for potential duplicate issue pairs:\n"
        "{enhanced_duplicate_insights}\n\n"
        "Synthesize this information into an executive summary that highlights the most critical findings, their relevance to the 2026 themes, and overarching recommendations.\n"
    )
)

# Initialize enhanced_cluster_details and enhanced_duplicate_insights if they are not defined
# This ensures the cell can run without NameError if previous cells haven't executed yet
if 'enhanced_cluster_details' not in locals() and 'enhanced_cluster_details' not in globals():
    enhanced_cluster_details = {}
if 'enhanced_duplicate_insights' not in locals() and 'enhanced_duplicate_insights' not in globals():
    enhanced_duplicate_insights = []

# Prepare cluster details and duplicate insights for the summary prompt
clusters_summary_str = ''
for cluster_id, details in enhanced_cluster_details.items():
    clusters_summary_str += f"- Cluster {cluster_id}: {details['name']}. Theme Alignment: {details['theme_alignment']}. Description: {details['description']}. PM Recommendations: {details['pm_recommendations']}\n"

duplicates_summary_str = ''
for insight in enhanced_duplicate_insights:
    duplicates_summary_str += (
        f"- Issues {insight['issue1_key']} ({insight['issue1_age_category']}) and {insight['issue2_key']} ({insight['issue2_age_category']}) (Similarity: {insight['similarity']:.4f}).\n"
        f"  Reasoning: {insight['llm_reasoning']}\n"
        f"  Recommendations: {insight['llm_recommendations']}\n"
    )

formatted_executive_summary_prompt = executive_summary_prompt_template.format(
    project_context=project_context,
    theme_2026_context=theme_2026_context,
    enhanced_cluster_details=clusters_summary_str,
    enhanced_duplicate_insights=duplicates_summary_str
)

executive_summary_response = llm.invoke(formatted_executive_summary_prompt)
executive_summary = executive_summary_response.content.strip()

print("\n--- Executive Summary ---\n")
print(executive_summary)

# Save the executive summary to a file
with open('executive_summary.txt', 'w') as f:
    f.write(executive_summary)
print("\nExecutive summary saved to 'executive_summary.txt'.")

In [ ]:
print("Generating Executive Summary...")

executive_summary_prompt_template = PromptTemplate(
    input_variables=["project_context", "theme_2026_context", "enhanced_cluster_details", "enhanced_duplicate_insights"],
    template=(
        "You are an expert in Jira backlog analysis and strategic reporting. Generate a concise executive summary (3-5 paragraphs) of the Jira issue analysis, focusing on key findings from cluster analysis and duplicate issue detection. Emphasize the alignment with and impact on the 2026 themes. Provide actionable, high-level recommendations for project management.\n\n"
        "## Project Context:\n{project_context}\n\n"
        "## 2026 Themes (PRIORITIZE for strategic alignment and impact):\n{theme_2026_context}\n\n"
        "## Cluster Analysis Summary:\n"
        "Based on the clustering of Jira issues, here are the enhanced details for each cluster:\n"
        "{enhanced_cluster_details}\n\n"
        "## Duplicate Issue Analysis Summary:\n"
        "Here are the insights for potential duplicate issue pairs:\n"
        "{enhanced_duplicate_insights}\n\n"
        "Synthesize this information into an executive summary that highlights the most critical findings, their relevance to the 2026 themes, and overarching recommendations.\n"
    )
)

# Prepare cluster details and duplicate insights for the summary prompt
clusters_summary_str = ''
for cluster_id, details in enhanced_cluster_details.items():
    clusters_summary_str += f"- Cluster {cluster_id}: {details['name']}. Theme Alignment: {details['theme_alignment']}. Description: {details['description']}. PM Recommendations: {details['pm_recommendations']}\n"

duplicates_summary_str = ''
for insight in enhanced_duplicate_insights:
    duplicates_summary_str += (
        f"- Issues {insight['issue1_key']} ({insight['issue1_age_category']}) and {insight['issue2_key']} ({insight['issue2_age_category']}) (Similarity: {insight['similarity']:.4f}).\n"
        f"  Reasoning: {insight['llm_reasoning']}\n"
        f"  Recommendations: {insight['llm_recommendations']}\n"
    )

formatted_executive_summary_prompt = executive_summary_prompt_template.format(
    project_context=project_context,
    theme_2026_context=theme_2026_context,
    enhanced_cluster_details=clusters_summary_str,
    enhanced_duplicate_insights=duplicates_summary_str
)

# Initialize the ChatGoogleGenAI model if not already done (for direct use here)
# NOTE: Hardcoding API keys is not recommended for production environments.
# This is done for demonstration purposes as per user request.
if 'llm' not in locals(): # Check if llm is already defined from previous cell
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0.3, api_key=api_key)

executive_summary_response = llm.invoke(formatted_executive_summary_prompt)
executive_summary = executive_summary_response.content.strip()

print("\n--- Executive Summary ---\n")
print(executive_summary)

# Save the executive summary to a file
with open('executive_summary.txt', 'w') as f:
    f.write(executive_summary)
print("\nExecutive summary saved to 'executive_summary.txt'.")

In [ ]:
executive_summary_content = """
Executive Summary: Jira Backlog Analysis

This report summarizes the findings from an in-depth analysis of Jira issues, leveraging advanced natural language processing (NLP), machine learning for clustering, and a Large Language Model (LLM) for enhanced insights.

## Cluster Analysis

Five distinct clusters of Jira issues were identified, each representing a core theme within the project backlog. The LLM provided descriptive names for these clusters:

"""

for cluster_id, name in cluster_names.items():
    executive_summary_content += f"- Cluster {cluster_id}: {name}\n"

executive_summary_content += """

## Duplicate Issue Analysis

Through cosine similarity, 12 potential duplicate issue pairs were identified, indicating areas for backlog streamlining. The LLM provided enhanced insights and actionable recommendations for each pair:

"""

for insight in enhanced_duplicate_insights:
    executive_summary_content += (
        f"- Issues {insight['issue1_key']} and {insight['issue2_key']} (Similarity: {insight['similarity']:.4f}):\n"
        f"  Reasoning: {insight['llm_reasoning']}\n"
        f"  Recommendations: {insight['llm_recommendations']}\n\n"
    )

executive_summary_content += """

## Conclusion

The clustering and duplicate detection processes, significantly enhanced by LLM-generated insights, provide a clearer, more organized view of the Jira backlog. These findings empower project managers and development teams to prioritize tasks more effectively, reduce redundant effort, and improve overall project efficiency.
"""

with open('executive_summary.txt', 'w') as f:
    f.write(executive_summary_content)

print("Generated 'executive_summary.txt' with the executive summary.")

In [ ]:
import matplotlib.pyplot as plt

# Assuming cluster_counts is already available from previous execution
# If not, it can be recalculated as: cluster_counts = df_jira_issues['cluster'].value_counts().sort_index()

# Create the bar plot
plt.figure(figsize=(10, 6))
cluster_counts.plot(kind='bar')

# Add labels and title
plt.xlabel('Cluster ID')
plt.ylabel('Number of Issues')
plt.title('Distribution of Issues Across Clusters')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Display the plot
plt.show()

In [ ]:
with open('executive_summary.txt', 'r') as f:
    summary_content = f.read()
print(summary_content)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate the number of issues per status
status_counts = df_jira_issues['status'].value_counts()

# Create the bar plot
plt.figure(figsize=(12, 7))
sns.barplot(x=status_counts.index, y=status_counts.values, palette='viridis')

# Add labels and title
plt.xlabel('Issue Status')
plt.ylabel('Number of Issues')
plt.title('Distribution of Jira Issues by Status')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Display the plot
plt.tight_layout()
plt.show()


In [ ]:
print("\nSummary Table: Issues per Cluster")
# cluster_counts is already available from previous execution
# If not, it can be recalculated as: cluster_counts = df_jira_issues['cluster'].value_counts().sort_index()
print(cluster_counts.to_string())

In [ ]:
print("\nRecommended Issues for Closing as Duplicates (with LLM insights):")
if enhanced_duplicate_insights:
    for insight in enhanced_duplicate_insights:
        print(f"- Issues {insight['issue1_key']} and {insight['issue2_key']} (Similarity: {insight['similarity']:.4f}):")
        print(f"  Reasoning: {insight['llm_reasoning']}")
        print(f"  Recommendations: {insight['llm_recommendations']}\n")
else:
    print("No potential duplicate issues found above the specified similarity threshold that require closing.")